# SunCedar MCP Toolkit · FastMCP + OpenAI Agents SDK
**EECE 503P/798S Agentic Systems · C5 Agent Tools** · Abbas Yassine

Three MCP tools for SunCedar Energy (fictional Beirut solar installer), one per category:

| Category | Tool | What it does |
|---|---|---|
| **Data** | `size_system` | Estimates daily kWh from appliances and recommends a package. Read-only. |
| **Action** | `book_site_visit` | Saves a site-visit booking and returns a booking ID. |
| **Orchestration** | `plan_solar_install` | Runs `size_system`, then `book_site_visit` with the recommended package. |

```
User → Agent (OpenAI Agents SDK = MCP Host) → MCP Client →(stdio, JSON-RPC)→ FastMCP server → tools
     ← final answer ←──────────────────────── tool result ←──────────────────────┘
```
Each cell below writes one file of the project, then the last cells run the tests and the agent.

In [ ]:
%pip install -q "fastmcp>=2.12" "openai-agents>=0.4" python-dotenv pytest

## 1. API key
Add `GEMINI_API_KEY` in **Secrets** (key icon on the left) with **Notebook access** on. If this cell says it timed out,
just run it again by itself (Colab sometimes times out on Secrets during *Run all*).

In [ ]:
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
print("Gemini key set:", bool(os.getenv("GEMINI_API_KEY")))

## 2. `core.py`: the business logic
Plain Python functions with no MCP code, so they are easy to test. Each user-fixable problem raises `ToolInputError`
with a clear message. The Orchestration function calls the other two in order and reports which step failed.

In [ ]:
%%writefile core.py
"""Business logic for the SunCedar MCP toolkit.

Plain Python functions with no MCP code, so they can be unit-tested directly.
server.py wraps them as MCP tools.
"""
from __future__ import annotations

import json
import re
import uuid
from datetime import datetime
from pathlib import Path

BOOKINGS_FILE = Path(__file__).resolve().parent / "data" / "bookings.json"

PACKAGES = {
    "Starter": {"price_usd": 2900, "panels_kw": 3, "battery_kwh": 5,
                "covers": "lights, fridge, router, TV"},
    "Family": {"price_usd": 4800, "panels_kw": 5, "battery_kwh": 10,
               "covers": "Starter loads plus a washing machine and a water pump"},
    "Comfort": {"price_usd": 7400, "panels_kw": 8, "battery_kwh": 15,
                "covers": "Family loads plus one inverter AC and an electric water heater"},
}

# appliance -> (peak watts, kWh per day): typical Lebanese home usage, planning estimates
APPLIANCES = {
    "lights": (120, 0.72), "fridge": (150, 1.50), "tv": (100, 0.50), "router": (15, 0.36),
    "laptop": (60, 0.36), "fan": (60, 0.48), "washing machine": (500, 0.50), "water pump": (750, 0.38),
    "microwave": (1000, 0.30), "inverter ac": (1000, 6.00), "electric water heater": (1500, 3.00),
}
ALIASES = {
    "light": "lights", "lamps": "lights", "refrigerator": "fridge", "television": "tv", "wifi": "router",
    "computer": "laptop", "washer": "washing machine", "pump": "water pump", "ac": "inverter ac",
    "air conditioner": "inverter ac", "aircon": "inverter ac", "water heater": "electric water heater",
    "heater": "electric water heater", "boiler": "electric water heater",
}
SERVED_TOWNS = {
    "beirut": "Greater Beirut", "hamra": "Greater Beirut", "achrafieh": "Greater Beirut",
    "jounieh": "Keserwan", "kaslik": "Keserwan", "baabda": "Baabda", "hazmieh": "Baabda",
    "broummana": "Metn", "antelias": "Metn", "aley": "Aley", "chouf": "Chouf",
    "jbeil": "Jbeil/Byblos", "byblos": "Jbeil/Byblos", "batroun": "Batroun",
}
EMAIL_RE = re.compile(r"^[^@\s]+@[^@\s]+\.[A-Za-z]{2,}$")


class ToolInputError(ValueError):
    """A predictable, user-fixable problem with the tool input."""


def _normalise(name: str) -> str | None:
    n = re.sub(r"^(a|an|the|my|\d+)\s+", "", name.strip().lower())
    if n in APPLIANCES:
        return n
    if n in ALIASES:
        return ALIASES[n]
    if n.endswith("s") and n[:-1] in ALIASES:
        return ALIASES[n[:-1]]
    return next((k for k in APPLIANCES if k in n), None)


# ---------------------------------------------------------------- DATA
def size_system(appliances: list[str]) -> dict:
    """Read-only: estimate daily energy use and recommend a home package."""
    if not appliances:
        raise ToolInputError("No appliances given. Example: ['fridge', 'lights', 'tv'].")
    known, unknown = {}, []
    for a in appliances:
        key = _normalise(a)
        (known.__setitem__(key, APPLIANCES[key]) if key else unknown.append(a))
    if not known:
        raise ToolInputError(f"None of these appliances are recognised: {unknown}. "
                             f"Known appliances: {sorted(APPLIANCES)}.")
    kwh = round(sum(v[1] for v in known.values()), 1)
    if "inverter ac" in known or "electric water heater" in known or kwh > 10:
        package = "Comfort"
    elif "washing machine" in known or "water pump" in known or kwh > 5:
        package = "Family"
    else:
        package = "Starter"
    if kwh > 15:
        raise ToolInputError(f"Estimated {kwh} kWh/day is above the largest home package (15 kWh battery). "
                             "This needs a Business Solar site survey.")
    return {"appliances": sorted(known), "unrecognised": unknown, "daily_kwh": kwh,
            "peak_watts": sum(v[0] for v in known.values()), "recommended_package": package,
            "price_usd": PACKAGES[package]["price_usd"]}


# ---------------------------------------------------------------- ACTION
def book_site_visit(name: str, email: str, town: str, package: str) -> dict:
    """Side effect: append a site-visit booking to data/bookings.json."""
    name, email, town = (name or "").strip(), (email or "").strip(), (town or "").strip()
    if not name:
        raise ToolInputError("Name is required.")
    if not EMAIL_RE.match(email):
        raise ToolInputError(f"'{email}' is not a valid email address.")
    if package not in PACKAGES:
        raise ToolInputError(f"Unknown package '{package}'. Choose one of {list(PACKAGES)}.")
    region = SERVED_TOWNS.get(town.lower())
    if region is None:
        raise ToolInputError(f"SunCedar does not serve '{town}' yet. Served areas: Greater Beirut, Metn, "
                             "Keserwan, Baabda, Aley, Chouf, Jbeil/Byblos and Batroun.")
    booking = {"booking_id": f"SV-{uuid.uuid4().hex[:6].upper()}", "name": name, "email": email,
               "town": town.title(), "region": region, "package": package,
               "created_at": datetime.now().isoformat(timespec="seconds"), "status": "pending_confirmation"}
    BOOKINGS_FILE.parent.mkdir(exist_ok=True)
    bookings = json.loads(BOOKINGS_FILE.read_text()) if BOOKINGS_FILE.exists() else []
    bookings.append(booking)
    BOOKINGS_FILE.write_text(json.dumps(bookings, indent=2))
    return booking


# ---------------------------------------------------------------- ORCHESTRATION
def plan_solar_install(appliances: list[str], name: str, email: str, town: str) -> dict:
    """Size the system, then book a site visit for the recommended package."""
    sizing = size_system(appliances)                      # step 1: data
    try:
        booking = book_site_visit(name, email, town, sizing["recommended_package"])   # step 2: action
    except ToolInputError as e:
        raise ToolInputError(f"Sizing succeeded ({sizing['recommended_package']}, {sizing['daily_kwh']} kWh/day) "
                             f"but the booking failed: {e}") from e
    return {"sizing": sizing, "booking": booking,
            "summary": f"{sizing['recommended_package']} package (from ${sizing['price_usd']:,}) for "
                       f"~{sizing['daily_kwh']} kWh/day; site visit {booking['booking_id']} requested in "
                       f"{booking['town']}."}


## 3. `server.py`: the FastMCP server
`@mcp.tool` for the three tools, `@mcp.resource` for the package catalogue (pure data). Arguments are typed, outputs
are Pydantic models (so FastMCP publishes output schemas), and failures become a clear `ToolError`. Every call is logged
to stderr as `[MCP server] ...` so the server side of the round trip is visible.

In [ ]:
%%writefile server.py
"""SunCedar MCP server (FastMCP): one Data tool, one Action tool, one Orchestration tool.

Run on its own:   python server.py          (stdio transport, which is what the agent uses)
"""
import sys
from typing import Literal

from fastmcp import FastMCP
from fastmcp.exceptions import ToolError
from pydantic import BaseModel, Field

import core

mcp = FastMCP(
    name="suncedar-solar",
    instructions="Tools for SunCedar Energy, a solar installer in Lebanon: size a home system, book a site visit, "
                 "or do both in one step.",
)


def log(msg: str) -> None:
    # stdout is the MCP channel, so server logs go to stderr (visible in the terminal during the demo)
    print(f"[MCP server] {msg}", file=sys.stderr, flush=True)


# ---------------------------------------------------------------- typed outputs
class Sizing(BaseModel):
    appliances: list[str] = Field(description="Appliances that were recognised")
    unrecognised: list[str] = Field(description="Inputs that were not recognised (ignored)")
    daily_kwh: float = Field(description="Estimated daily energy use in kWh")
    peak_watts: int = Field(description="Sum of appliance peak power in watts")
    recommended_package: Literal["Starter", "Family", "Comfort"]
    price_usd: int = Field(description="Package price from (USD, installed)")


class Booking(BaseModel):
    booking_id: str
    name: str
    email: str
    town: str
    region: str
    package: Literal["Starter", "Family", "Comfort"]
    created_at: str
    status: str


class InstallPlan(BaseModel):
    sizing: Sizing
    booking: Booking
    summary: str


# ---------------------------------------------------------------- resource (pure data lookup)
@mcp.resource("suncedar://packages", name="packages", description="SunCedar home packages and prices.",
              mime_type="application/json")
def packages() -> dict:
    return core.PACKAGES


# ---------------------------------------------------------------- DATA tool
@mcp.tool
def size_system(appliances: list[str]) -> Sizing:
    """DATA tool (read-only). Estimate a home's daily energy use from a list of appliances and recommend
    one SunCedar package (Starter, Family or Comfort). Changes nothing.
    Example input: ["fridge", "lights", "tv", "washing machine"]."""
    log(f"size_system(appliances={appliances})")
    try:
        return Sizing(**core.size_system(appliances))
    except core.ToolInputError as e:
        log(f"  -> error: {e}")
        raise ToolError(str(e)) from e


# ---------------------------------------------------------------- ACTION tool
@mcp.tool
def book_site_visit(name: str, email: str, town: str,
                    package: Literal["Starter", "Family", "Comfort"]) -> Booking:
    """ACTION tool (side effect). Book a free on-site survey for one package. Saves the booking to
    data/bookings.json and returns its booking_id. Fails clearly if the email is invalid or the town is not served.
    Only call this when the customer has given their name, email and town."""
    log(f"book_site_visit(name={name!r}, email={email!r}, town={town!r}, package={package!r})")
    try:
        result = Booking(**core.book_site_visit(name, email, town, package))
        log(f"  -> saved {result.booking_id}")
        return result
    except core.ToolInputError as e:
        log(f"  -> error: {e}")
        raise ToolError(str(e)) from e


# ---------------------------------------------------------------- ORCHESTRATION tool
@mcp.tool
def plan_solar_install(appliances: list[str], name: str, email: str, town: str) -> InstallPlan:
    """ORCHESTRATION tool. For a customer who lists appliances AND wants a visit: runs size_system, then
    book_site_visit with the recommended package, and returns both results with a one-line summary.
    If booking fails after sizing, the error says which step failed."""
    log(f"plan_solar_install(appliances={appliances}, name={name!r}, email={email!r}, town={town!r})")
    try:
        result = InstallPlan(**core.plan_solar_install(appliances, name, email, town))
        log(f"  -> step 1 size_system: {result.sizing.recommended_package}; "
            f"step 2 book_site_visit: {result.booking.booking_id}")
        return result
    except core.ToolInputError as e:
        log(f"  -> error: {e}")
        raise ToolError(str(e)) from e


if __name__ == "__main__":
    mcp.run(show_banner=False)  # stdio


## 4. `agent.py`: the OpenAI Agents SDK agent (MCP Host/Client)
`MCPServerStdio` starts `server.py` as a subprocess and connects to it; the agent discovers the tools and decides which
one to call. The model is Gemini through its OpenAI-compatible endpoint. The host prints each call and result, so the
full round trip agent → MCP client → server → agent is visible.

In [ ]:
%%writefile agent.py
"""SunCedar agent: OpenAI Agents SDK as the MCP Host/Client, connected to server.py over stdio.

  python agent.py --list-tools   # connect to the MCP server and list its tools (no API key needed)
  python agent.py --demo         # run the scripted demo queries
  python agent.py                # interactive chat

Model: Gemini through its OpenAI-compatible endpoint (GEMINI_API_KEY), or OpenAI (OPENAI_API_KEY).
"""
import argparse
import asyncio
import json
import os
import sys
from pathlib import Path

from agents import Agent, Runner, OpenAIChatCompletionsModel, set_tracing_disabled
from agents.items import ToolCallItem, ToolCallOutputItem
from agents.mcp import MCPServerStdio

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

HERE = Path(__file__).resolve().parent
GEMINI_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GEMINI_MODELS = [m for m in [os.getenv("GEMINI_MODEL"), "gemini-3.5-flash-lite", "gemini-3.1-flash-lite",
                             "gemini-3.8-flash"] if m]

INSTRUCTIONS = """You are SunCedar Energy's solar assistant for homes in Lebanon.
Use the MCP tools, never guess numbers:
- Customer lists appliances or asks which package fits, and does NOT ask for a visit -> size_system.
- Customer already knows the package and wants a visit -> book_site_visit.
- Customer lists appliances AND wants a visit in one request -> plan_solar_install (it sizes, then books).
- Never invent a name, email or town. If one is missing for a booking, ask for it.
- If a tool returns an error, explain it plainly and say what the customer can do.
Keep answers short (2-4 sentences) and mention the booking ID when a visit is booked."""

DEMO = [
    "I run a fridge, lights, a TV and a washing machine. Which package do I need?",
    "Please book a site visit for the Comfort package. I'm Abbas Yassine, abbasyassine@example.com, in Jounieh.",
    "I have an inverter AC, a fridge and lights. Size my system and book a visit: Abbas Yassine, "
    "abbasyassine@example.com, Hamra.",
    "Book the Starter package for me in Zahle. Name Abbas Yassine, email abbasyassine@example.com.",
]


def make_model(index: int = 0):
    if os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY"):
        from openai import AsyncOpenAI
        client = AsyncOpenAI(api_key=os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY"),
                             base_url=GEMINI_URL)
        name = GEMINI_MODELS[min(index, len(GEMINI_MODELS) - 1)]
        return OpenAIChatCompletionsModel(model=name, openai_client=client), name
    if os.getenv("OPENAI_API_KEY"):
        name = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
        return name, name
    sys.exit("No API key: set GEMINI_API_KEY (or OPENAI_API_KEY) in .env or the environment.")


def server_params():
    env = {**os.environ, "FASTMCP_LOG_LEVEL": "ERROR", "PYTHONUNBUFFERED": "1"}  # quiet server banner/logs
    return {"command": sys.executable, "args": [str(HERE / "server.py")], "cwd": str(HERE), "env": env}


def show_round_trip(result) -> None:
    """Print each tool call the agent made and what the MCP server returned."""
    for item in result.new_items:
        if isinstance(item, ToolCallItem):
            raw = item.raw_item
            name = getattr(raw, "name", None) or (raw.get("name") if isinstance(raw, dict) else "?")
            args = getattr(raw, "arguments", None) or (raw.get("arguments") if isinstance(raw, dict) else "")
            print(f"  [agent -> MCP client -> server] call {name}({args})")
        elif isinstance(item, ToolCallOutputItem):
            out = item.output if isinstance(item.output, str) else json.dumps(item.output)
            print(f"  [server -> MCP client -> agent] {out[:300]}")


async def ask(server, question: str, history: list, model_index: int = 0):
    model, name = make_model(model_index)
    agent = Agent(name="SunCedar assistant", instructions=INSTRUCTIONS, mcp_servers=[server], model=model)
    try:
        return await Runner.run(agent, history + [{"role": "user", "content": question}], max_turns=8), name
    except Exception as e:
        text = str(e)
        if model_index + 1 < len(GEMINI_MODELS) and ("404" in text or "429" in text or "RESOURCE_EXHAUSTED" in text):
            print(f"  (model {name} unavailable or over quota, trying the next one)")
            return await ask(server, question, history, model_index + 1)
        raise


async def main(mode: str) -> None:
    set_tracing_disabled(True)  # tracing would need an OpenAI key
    async with MCPServerStdio(params=server_params(), name="suncedar-solar", cache_tools_list=True,
                              client_session_timeout_seconds=30) as server:
        tools = await server.list_tools()
        print(f"Connected to MCP server 'suncedar-solar' with {len(tools)} tools: {[t.name for t in tools]}\n")
        if mode == "list":
            for t in tools:
                print(f"- {t.name}: {t.description.splitlines()[0]}")
                print(f"  input schema: {json.dumps((getattr(t, 'inputSchema', None) or getattr(t, 'input_schema', {})).get('properties', {}))}")
            return
        history = []
        questions = DEMO if mode == "demo" else None
        while True:
            if questions is not None:
                if not questions:
                    break
                q = questions.pop(0)
                print(f"USER: {q}")
            else:
                q = input("USER: ").strip()
                if q.lower() in ("", "quit", "exit"):
                    break
            result, model_name = await ask(server, q, history)
            show_round_trip(result)
            print(f"AGENT ({model_name}): {result.final_output}\n")
            history = result.to_input_list() if mode != "demo" else []


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--demo", action="store_true")
    p.add_argument("--list-tools", action="store_true")
    a = p.parse_args()
    asyncio.run(main("list" if a.list_tools else "demo" if a.demo else "chat"))


## 5. `tests/test_tools.py`: tests for each tool, each failure case, and the MCP layer

In [ ]:
import os; os.makedirs('tests', exist_ok=True)

In [ ]:
%%writefile tests/test_tools.py
"""Tests for the three tools: core logic, the MCP layer (in-memory FastMCP client), and the failure cases.

Run:  python -m pytest -v
"""
import asyncio
import json
import sys
from pathlib import Path

import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
import core      # noqa: E402
import server    # noqa: E402
from fastmcp import Client                 # noqa: E402
from fastmcp.exceptions import ToolError   # noqa: E402


@pytest.fixture(autouse=True)
def temp_bookings(tmp_path, monkeypatch):
    """Write bookings to a temporary file so tests never touch data/bookings.json."""
    monkeypatch.setattr(core, "BOOKINGS_FILE", tmp_path / "bookings.json")
    return tmp_path / "bookings.json"


def call(tool, args):
    async def _run():
        async with Client(server.mcp) as c:
            return await c.call_tool(tool, args)
    return asyncio.run(_run())


# ---------------------------------------------------------------- DATA tool
def test_size_system_family():
    r = core.size_system(["fridge", "lights", "TV", "washing machine"])
    assert r["recommended_package"] == "Family" and r["daily_kwh"] == 3.2


def test_size_system_ac_needs_comfort():
    assert core.size_system(["an AC", "fridge"])["recommended_package"] == "Comfort"


def test_size_system_is_read_only(temp_bookings):
    core.size_system(["fridge"])
    assert not temp_bookings.exists()


def test_size_system_failure_unknown_appliances():
    with pytest.raises(core.ToolInputError, match="None of these appliances are recognised"):
        core.size_system(["spaceship", "time machine"])


def test_size_system_failure_empty():
    with pytest.raises(core.ToolInputError):
        core.size_system([])


# ---------------------------------------------------------------- ACTION tool
def test_book_site_visit_writes_booking(temp_bookings):
    b = core.book_site_visit("Abbas Yassine", "abbasyassine@example.com", "Jounieh", "Comfort")
    saved = json.loads(temp_bookings.read_text())
    assert b["booking_id"].startswith("SV-") and saved[0]["booking_id"] == b["booking_id"]
    assert saved[0]["region"] == "Keserwan"


def test_book_site_visit_failure_unserved_town(temp_bookings):
    with pytest.raises(core.ToolInputError, match="does not serve 'Zahle'"):
        core.book_site_visit("Abbas", "abbasyassine@example.com", "Zahle", "Starter")
    assert not temp_bookings.exists()   # nothing written on failure


def test_book_site_visit_failure_bad_email():
    with pytest.raises(core.ToolInputError, match="not a valid email"):
        core.book_site_visit("Abbas", "not-an-email", "Beirut", "Starter")


# ---------------------------------------------------------------- ORCHESTRATION tool
def test_plan_solar_install_runs_both_steps(temp_bookings):
    r = core.plan_solar_install(["inverter ac", "fridge", "lights"], "Abbas", "abbasyassine@example.com", "Hamra")
    assert r["sizing"]["recommended_package"] == "Comfort"
    assert r["booking"]["package"] == "Comfort"            # booked what sizing recommended
    assert len(json.loads(temp_bookings.read_text())) == 1


def test_plan_solar_install_failure_says_which_step(temp_bookings):
    with pytest.raises(core.ToolInputError, match="Sizing succeeded .* but the booking failed"):
        core.plan_solar_install(["fridge"], "Abbas", "abbasyassine@example.com", "Tripoli")
    assert not temp_bookings.exists()


# ---------------------------------------------------------------- MCP layer (what the agent actually sees)
def test_server_exposes_exactly_three_tools_and_a_resource():
    async def _run():
        async with Client(server.mcp) as c:
            return await c.list_tools(), await c.list_resources()
    tools, resources = asyncio.run(_run())
    assert sorted(t.name for t in tools) == ["book_site_visit", "plan_solar_install", "size_system"]
    assert all(t.description and (getattr(t, "input_schema", None) or getattr(t, "inputSchema", None)) for t in tools)
    assert all(getattr(t, "output_schema", None) or getattr(t, "outputSchema", None) for t in tools)             # typed outputs
    assert [str(r.uri) for r in resources] == ["suncedar://packages"]


def test_mcp_call_returns_structured_output():
    r = call("size_system", {"appliances": ["fridge", "lights", "tv"]})
    assert r.structured_content["recommended_package"] == "Starter"


def test_mcp_failure_is_a_clear_tool_error():
    with pytest.raises(ToolError, match="does not serve"):
        call("book_site_visit", {"name": "Abbas", "email": "abbasyassine@example.com", "town": "Zahle",
                                 "package": "Starter"})


def test_mcp_rejects_invalid_package_value():
    with pytest.raises(ToolError):
        call("book_site_visit", {"name": "Abbas", "email": "abbasyassine@example.com", "town": "Beirut",
                                 "package": "Platinum"})


## 6. Run the tests

In [ ]:
!python -m pytest -v

## 7. The agent connects to the MCP server and discovers the tools (no LLM call)

In [ ]:
!FASTMCP_LOG_LEVEL=ERROR python -u agent.py --list-tools 2>&1 | cat

## 8. Demo: the agent using all three tools, plus a failure case
1. Appliances only → **Data** tool · 2. Known package + details → **Action** tool ·
3. Appliances + visit in one request → **Orchestration** tool · 4. Town not served → clear error

In [ ]:
!FASTMCP_LOG_LEVEL=ERROR python -u agent.py --demo 2>&1 | cat

## 9. The Action tool's side effect: the saved bookings

In [ ]:
!cat data/bookings.json